In [12]:
import sys
import os

sys.path.append(os.path.abspath("../.."))
from src.config.llm import GoogleLLm
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,Annotated
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage,SystemMessage

In [13]:
llm_config = GoogleLLm()
model = llm_config.llm


In [14]:
#state 
class TweetState(TypedDict):
    topic:str
    tweet:str
    evalution:Literal['approved','needs_improvement']
    feedback:str
    iteration:int
    max_iterator:int #for break the loop 

class EvaluationSchema(BaseModel):
    evaluation:Literal['approved','needs_improvement'] = Field(..., description="Final evaluation Result")
    feedback:str=Field(..., description="feedback for the tweet")
    

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [9]:
def generate_tweet(state:TweetState):
    #prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]
    response = model.invoke(messages).content
    return{'tweet':response}

    

In [15]:
def evaluate_tweet(state:TweetState):
    tweet = state['tweet']
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{tweet}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    response = structured_model.invoke(messages)
    return{'evaluation':response.evaluation,'feedback':response.feedback}

In [ ]:
def optimize_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]


In [ ]:
graph = StateGraph(TweetState)

#nodes
graph.add_node("generate", generate_tweet)
graph.add_node("evaluate",evaluate_tweet)
graph.add_node("optimize",optimize_tweet)